# Adapter Negation / Task Arithmetic — TOFU Benchmark (Gemma 3 4B IT)

**Method**: Task Arithmetic (Ilharco et al., 2023)  
**Dataset**: [TOFU](https://huggingface.co/datasets/locuslab/TOFU) — Task of Fictitious Unlearning  
**Model**: Gemma 3 4B Instruct with TOFU LoRA adapter  
**Platform**: Kaggle T4 × 2

**Pipeline**:
1. Train a LoRA adapter on the **forget set** using the **base** (pre-fine-tuned) model → captures $\Delta_{\text{forget}}$
2. Load the fine-tuned model $\theta_{\text{ft}}$ (base + TOFU adapter merged)
3. Subtract: $\theta_{\text{unlearned}} = \theta_{\text{ft}} - \lambda \cdot \Delta_{\text{forget}}$
4. Evaluate FSR (Forget Success Rate) and RR (Retention Rate)


In [ ]:
# ── Install dependencies ──
!pip install -q transformers peft accelerate safetensors bitsandbytes>=0.46.1 datasets torchao --upgrade


In [ ]:
# ── Imports & Auth ──
import torch, gc, os, json
from collections import defaultdict
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
    TrainingArguments, Trainer, DataCollatorForSeq2Seq,
)
from peft import PeftModel, LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from datasets import load_dataset, Dataset
from tqdm.notebook import tqdm
from huggingface_hub import login, HfApi

HF_TOKEN = ""
login(token=HF_TOKEN)

os.environ.pop("TQDM_DISABLE", None)

from transformers import logging as hf_logging
hf_logging.set_verbosity_info()
hf_logging.enable_progress_bar()

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count:      {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name}  {props.total_memory/1024**3:.1f} GiB")


## Configuration

- **FORGET_SPLIT**: `forget05` (5%)
- **RETAIN_SPLIT**: `retain95` (95%)
- **LAMBDA**: Fixed at 1.0 (Task Arithmetic default)


In [ ]:
# ── Configuration ──
MODEL_CONFIG = {
    "base_model": "google/gemma-3-4b-it",
    "adapter":    "Novaspree/tofu-Gemma3-adapter-1",   # fine-tuned TOFU LoRA
    "name":       "Gemma-3-4B-IT",
}

# TOFU split — forget05 / retain95
FORGET_SPLIT = "forget05"
RETAIN_SPLIT = "retain95"

# Task Arithmetic lambda (fixed at 1.0 as specified)
LAMBDA          = 1.0
MAX_NEW_TOKENS  = 64
MAX_SEQ_LENGTH  = 256

# Where to save the forget adapter
FORGET_ADAPTER_LOCAL = "./tofu_gemma3_forget_adapter"
FORGET_ADAPTER_HF    = "Novaspree/tofu-gemma3-forget-adapter-05"

# Predictions repo
PRED_REPO      = "Novaspree/tofu-gemma3-adapter-negation-predictions"
LOCAL_PRED_DIR = "./tofu_predictions"

# Training hyperparameters for the forget adapter
NUM_EPOCHS       = 5
BATCH_SIZE       = 4
GRAD_ACCUM_STEPS = 2
LEARNING_RATE    = 2e-4

print("Configuration loaded ✓")
print(f"  Model:   {MODEL_CONFIG['name']}")
print(f"  Adapter: {MODEL_CONFIG['adapter']}")
print(f"  Split:   {FORGET_SPLIT} / {RETAIN_SPLIT}")
print(f"  λ:       {LAMBDA}")


In [ ]:
# ── Load TOFU datasets ──
forget_ds_hf = load_dataset("locuslab/TOFU", FORGET_SPLIT, split="train")
retain_ds_hf = load_dataset("locuslab/TOFU", RETAIN_SPLIT, split="train")

forget_set = [{"question": r["question"], "answer": r["answer"]} for r in forget_ds_hf]
retain_set = [{"question": r["question"], "answer": r["answer"]} for r in retain_ds_hf]

print(f"TOFU {FORGET_SPLIT}/{RETAIN_SPLIT} loaded:")
print(f"  Forget : {len(forget_set)} samples")
print(f"  Retain : {len(retain_set)} samples")
print(f"\nExample forget sample:")
print(f"  Q: {forget_set[0]['question']}")
print(f"  A: {forget_set[0]['answer'][:100]}...")


## Step 1 — Train Forget Adapter

Train a LoRA adapter on the **forget set** using the **base** (pre-fine-tuned) model.  
Captures the "forget direction" $\Delta_{\text{forget}}$ to be subtracted later.

> **Gemma 3 4B IT notes**  
> - Attention modules: `q_proj`, `k_proj`, `v_proj`, `o_proj`  
> - MLP modules: `gate_proj`, `up_proj`, `down_proj`  
> - All 26 transformer layers targeted (no layer restriction), ensuring full coverage.  
> - 4-bit QLoRA via BitsAndBytes on both T4 GPUs (`device_map="auto"`).


In [ ]:
# ── STEP 1: Train forget adapter on TOFU forget set ──────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,   # Gemma 3 is bfloat16-native
)

print(f"Loading base model in 4-bit QLoRA: {MODEL_CONFIG['base_model']}")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_CONFIG["base_model"],
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    token=HF_TOKEN,
)
base_model = prepare_model_for_kbit_training(base_model)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_CONFIG["base_model"],
    trust_remote_code=True,
    token=HF_TOKEN,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# LoRA config for Gemma 3 4B IT — all layers, full module coverage
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    # No layers_to_transform → all 26 layers targeted
    task_type=TaskType.CAUSAL_LM,
    bias="none",
)

model = get_peft_model(base_model, lora_config)
print("Forget adapter created:")
model.print_trainable_parameters()

# ── Tokenize forget set ──
def tokenize_function(examples):
    texts = [
        f"Question: {q}\nAnswer: {a}"
        for q, a in zip(examples["question"], examples["answer"])
    ]
    tokenized = tokenizer(
        texts, truncation=True, max_length=MAX_SEQ_LENGTH, padding=False
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

forget_ds_tok = Dataset.from_list(forget_set)
forget_ds_tok = forget_ds_tok.map(
    tokenize_function, batched=True, remove_columns=forget_ds_tok.column_names
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer, pad_to_multiple_of=8, return_tensors="pt", padding=True,
)

training_args = TrainingArguments(
    output_dir=FORGET_ADAPTER_LOCAL,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    bf16=True,          # bfloat16 for Gemma 3 (T4 supports bf16)
    fp16=False,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    disable_tqdm=False,
)

from transformers import TrainerCallback

class ProgressCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            print(f"  Step {state.global_step}/{state.max_steps} "
                  f"— loss: {logs.get('loss', 'N/A')}, "
                  f"lr: {logs.get('learning_rate', 'N/A'):.2e}"
                  if isinstance(logs.get('learning_rate'), float)
                  else f"  Step {state.global_step}/{state.max_steps} — {logs}")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=forget_ds_tok,
    data_collator=data_collator,
    callbacks=[ProgressCallback()],
)

print(f"\nTraining forget adapter on {len(forget_set)} TOFU samples ({FORGET_SPLIT})...")
trainer.train()
print("✓ Training complete!")


In [ ]:
# ── Save & upload forget adapter ──
model.save_pretrained(FORGET_ADAPTER_LOCAL)
tokenizer.save_pretrained(FORGET_ADAPTER_LOCAL)
print(f"✓ Saved locally → {FORGET_ADAPTER_LOCAL}")

api = HfApi()
api.create_repo(FORGET_ADAPTER_HF, exist_ok=True, token=HF_TOKEN)
api.upload_folder(
    folder_path=FORGET_ADAPTER_LOCAL,
    repo_id=FORGET_ADAPTER_HF,
    token=HF_TOKEN,
)
print(f"✓ Uploaded → https://huggingface.co/{FORGET_ADAPTER_HF}")


In [ ]:
# ── Cleanup VRAM after Step 1 ──
del model, base_model, trainer
gc.collect()
torch.cuda.empty_cache()

for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"  GPU {i}: {free/1024**3:.1f} / {total/1024**3:.1f} GiB free")


## Steps 2 & 3 — Load Fine-Tuned Model & Subtract Forget Adapter

Quick single-λ test (λ = 1.0) before the full sweep.

**Formula**: $\theta_{\text{unlearned}} = \theta_{\text{ft}} - \lambda \cdot \Delta_{\text{forget}}$  

We negate `lora_B` weights by `−λ` before merging, which is equivalent to  
subtracting the adapter's contribution from the merged weights.


In [ ]:
# ── STEP 2 & 3: Load fine-tuned model + subtract forget adapter ──────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_CONFIG["base_model"], trust_remote_code=True, token=HF_TOKEN,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 1. Load base + merge TOFU adapter → θ_ft
print(f"Loading base model: {MODEL_CONFIG['base_model']}")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_CONFIG["base_model"],
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    token=HF_TOKEN,
)

print(f"Merging TOFU adapter: {MODEL_CONFIG['adapter']}")
ft_model = PeftModel.from_pretrained(base_model, MODEL_CONFIG["adapter"], token=HF_TOKEN)
ft_model = ft_model.merge_and_unload()
print("✓ Fine-tuned model ready (TOFU knowledge baked in).")

# 2. Load forget adapter onto ft_model
print(f"\nLoading forget adapter from: {FORGET_ADAPTER_HF}")
model = PeftModel.from_pretrained(ft_model, FORGET_ADAPTER_HF, token=HF_TOKEN)

# 3. Negate lora_B by −λ  →  effective subtraction of Δ_forget
negated_count = 0
for name, param in model.named_parameters():
    if "lora_B" in name and "weight" in name:
        param.data.mul_(-LAMBDA)
        negated_count += 1

# 4. Merge & unload for fast inference
model = model.merge_and_unload()
model.eval()

print(f"\n{'='*52}")
print(f"  FORGET ADAPTER NEGATION COMPLETE")
print(f"  λ = {LAMBDA}")
print(f"  lora_B matrices negated: {negated_count}")
print(f"  θ_unlearned = θ_ft − {LAMBDA} · Δ_forget")
print(f"{'='*52}")


## Evaluation

| Metric | Definition | Target |
|--------|-----------|--------|
| **FSR** (Forget Success Rate) | Fraction of forget-set samples model **fails** to recall | ↑ Higher = better unlearning |
| **RR** (Retention Rate) | Fraction of retain-set samples model **still recalls** | ↑ Higher = less collateral damage |


In [ ]:
# ── Batched evaluation function ──────────────────────────────────────────────
@torch.no_grad()
def evaluate_model_batched(model, tokenizer, dataset, split_name="forget", batch_size=8):
    """Exact-match accuracy with batched generation (left-padded)."""
    model.eval()
    correct, total = 0, 0
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    for i in tqdm(range(0, len(dataset), batch_size), desc=f"Eval [{split_name}]"):
        batch   = dataset[i : i + batch_size]
        prompts = [f"Question: {item['question']}\nAnswer:" for item in batch]
        targets = [item["answer"].strip().lower() for item in batch]

        inputs = tokenizer(
            prompts, return_tensors="pt", padding=True,
            truncation=True, max_length=MAX_SEQ_LENGTH,
        ).to(model.device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

        input_len = inputs["input_ids"].shape[1]
        for output, target in zip(outputs, targets):
            generated = tokenizer.decode(
                output[input_len:], skip_special_tokens=True
            ).strip().lower()
            if target in generated:
                correct += 1
            total += 1

    tokenizer.padding_side = "right"
    return correct / total if total > 0 else 0.0


In [ ]:
# ── Single-λ evaluation (λ = 1.0) ───────────────────────────────────────────
print(f"MODEL  : {MODEL_CONFIG['name']}")
print(f"METHOD : Adapter Negation (Task Arithmetic, λ={LAMBDA})")
print(f"DATASET: TOFU {FORGET_SPLIT}/{RETAIN_SPLIT}")
print()

forget_recall = evaluate_model_batched(model, tokenizer, forget_set, "forget")
fsr           = 1.0 - forget_recall
print(f"\n  FORGET → FSR = {fsr:.1%}  (recall rate = {forget_recall:.1%})")

retain_recall = evaluate_model_batched(model, tokenizer, retain_set, "retain")
rr            = retain_recall
print(f"  RETAIN → RR  = {rr:.1%}")

print(f"\n{'='*40}")
print(f"  FSR (↑ better): {fsr:.1%}")
print(f"  RR  (↑ better): {rr:.1%}")
print(f"{'='*40}")


## Collect & Upload Predictions


In [ ]:
# ── Collect full predictions ─────────────────────────────────────────────────
@torch.no_grad()
def collect_predictions(model, tokenizer, dataset, split_name="forget"):
    """Inference + collect (question, answer, generation, match)."""
    model.eval()
    records = []
    tokenizer.padding_side = "left"

    for item in tqdm(dataset, desc=f"{split_name} set"):
        question      = item["question"]
        target_answer = item["answer"].strip()
        prompt        = f"Question: {question}\nAnswer:"

        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
        generated = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        ).strip()

        records.append({
            "question":        question,
            "target_answer":   target_answer,
            "model_generation": generated,
            "exact_match":     target_answer.lower() in generated.lower(),
            "split":           split_name,
            "model":           MODEL_CONFIG["name"],
            "method":          "adapter_negation",
            "lambda":          LAMBDA,
            "dataset":         "TOFU",
            "forget_split":    FORGET_SPLIT,
        })

    tokenizer.padding_side = "right"
    print(f"  ✓ {len(records)} predictions collected")
    return records

forget_preds = collect_predictions(model, tokenizer, forget_set, "forget")
retain_preds = collect_predictions(model, tokenizer, retain_set, "retain")
all_preds    = forget_preds + retain_preds
print(f"\nTotal: {len(all_preds)} ({len(forget_preds)} forget + {len(retain_preds)} retain)")


In [ ]:
# ── Upload predictions to HuggingFace ────────────────────────────────────────
os.makedirs(LOCAL_PRED_DIR, exist_ok=True)

pred_file = f"{LOCAL_PRED_DIR}/predictions_{FORGET_SPLIT}_lambda{LAMBDA}.json"
with open(pred_file, "w") as f:
    json.dump(all_preds, f, indent=2)
print(f"✓ Saved {len(all_preds)} predictions → {pred_file}")

api = HfApi()
api.create_repo(PRED_REPO, repo_type="dataset", exist_ok=True, token=HF_TOKEN)
api.upload_folder(
    folder_path=LOCAL_PRED_DIR,
    repo_id=PRED_REPO,
    repo_type="dataset",
    token=HF_TOKEN,
)
print(f"✓ Uploaded → https://huggingface.co/datasets/{PRED_REPO}")


## Lambda Sweep (Optional)

Sweep over different λ values to map the FSR / RR trade-off curve.  
The forget adapter is already trained — this section **only reloads & evaluates**.

> Re-run the **Imports & Auth** and **Configuration** cells if the runtime was restarted.


In [ ]:
# ── Lambda Sweep ─────────────────────────────────────────────────────────────
LAMBDA_VALUES = [0.25, 0.5, 1.0, 1.5, 2.0]
sweep_results = []

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_CONFIG["base_model"], trust_remote_code=True, token=HF_TOKEN,
)
if _tokenizer.pad_token is None:
    _tokenizer.pad_token = _tokenizer.eos_token

for lam in LAMBDA_VALUES:
    print(f"\n{'='*52}\nSWEEP: λ = {lam}\n{'='*52}")

    # Fresh base model
    _base = AutoModelForCausalLM.from_pretrained(
        MODEL_CONFIG["base_model"],
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        token=HF_TOKEN,
    )

    # Merge TOFU adapter → θ_ft
    _ft = PeftModel.from_pretrained(_base, MODEL_CONFIG["adapter"], token=HF_TOKEN)
    _ft = _ft.merge_and_unload()

    # Load forget adapter & negate with this λ
    _m = PeftModel.from_pretrained(_ft, FORGET_ADAPTER_HF, token=HF_TOKEN)
    for name, param in _m.named_parameters():
        if "lora_B" in name and "weight" in name:
            param.data.mul_(-lam)
    _m = _m.merge_and_unload()
    _m.eval()

    f_rate = evaluate_model_batched(_m, _tokenizer, forget_set, "forget")
    r_rate = evaluate_model_batched(_m, _tokenizer, retain_set, "retain")
    fsr_i  = 1.0 - f_rate

    print(f"λ={lam:.2f}  FSR={fsr_i:.1%}  RR={r_rate:.1%}")
    sweep_results.append({"lambda": lam, "FSR": fsr_i, "RR": r_rate})

    del _m, _ft, _base
    gc.collect()
    torch.cuda.empty_cache()

# ── Summary ──
print(f"\n{'='*52}")
print(f"LAMBDA SWEEP SUMMARY — TOFU {FORGET_SPLIT}/{RETAIN_SPLIT}")
print(f"Model: {MODEL_CONFIG['name']}")
print(f"Forget: {len(forget_set)} samples | Retain: {len(retain_set)} samples")
print(f"{'='*52}")
print(f"{'λ':>6}  {'FSR (↑)':>10}  {'RR (↑)':>10}")
print("-" * 30)
for r in sweep_results:
    print(f"{r['lambda']:>6.2f}  {r['FSR']:>10.1%}  {r['RR']:>10.1%}")
best = max(sweep_results, key=lambda x: x["FSR"] + x["RR"])
print(f"\n★ Best balanced λ = {best['lambda']} "
      f"(FSR={best['FSR']:.1%}, RR={best['RR']:.1%})")

# ── Save sweep results ──
os.makedirs(LOCAL_PRED_DIR, exist_ok=True)
sweep_file = f"{LOCAL_PRED_DIR}/lambda_sweep_{FORGET_SPLIT}.json"
with open(sweep_file, "w") as f:
    json.dump(sweep_results, f, indent=2)

api = HfApi()
api.create_repo(PRED_REPO, repo_type="dataset", exist_ok=True, token=HF_TOKEN)
api.upload_folder(
    folder_path=LOCAL_PRED_DIR,
    repo_id=PRED_REPO,
    repo_type="dataset",
    token=HF_TOKEN,
)
print(f"\n✓ Sweep results uploaded → https://huggingface.co/datasets/{PRED_REPO}")
